In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_68_try_3.pickle

In [ ]:
%%RecordEvent
%%time
### cell 69 ###

# Standardize and rename the 2019 responses in one GPU call
responses_df_2019_cell10 = responses_df_2019_cell10.rename(
    columns={tq_2019: tq_2022}
)
responses_df_2019_cell10[tq_2022] = responses_df_2019_cell10[tq_2022].replace({
    '6-24 times': '6-25 times',
    '> 25 times': 'More than 25 times'
})

# Prepare list of (df, year) pairs
df_list = [
    (responses_df_2019_cell10, '2019'),
    (responses_df_2020,       '2020'),
    (responses_df_2021,       '2021'),
    (responses_df_2022_cell10,'2022')
]

# Concatenate all years and assign the year column in one GPU‐friendly step
combined = pd.concat(
    [df[[question_of_interest]].assign(year=year) for df, year in df_list],
    ignore_index=True
)

# Compute counts per category and year on the GPU
gpu_counts = (
    combined
    .groupby([question_of_interest, 'year'])
    .size()
    .reset_index(name='count')
)

# Compute total counts per year and merge back
totals = gpu_counts.groupby('year')['count'].sum().reset_index(name='total')
counts_with_totals = gpu_counts.merge(totals, on='year')

# Calculate percentage and round
gpu_counts_final = counts_with_totals.assign(
    percentage=(counts_with_totals['count'] * 100 / counts_with_totals['total']).round(1)
)

# Final formatting: rename, select, sort
x_axis_title_cell81 = 'Frequency of TPU usage'
df_combined_cell81 = (
    gpu_counts_final[[question_of_interest, 'percentage', 'year']]
    .rename(columns={question_of_interest: x_axis_title_cell81})
    .sort_values(by=['year', 'percentage'], ascending=True)
)

# Display result
df_combined_cell81

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_69_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_69_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[69], f)


In [ ]:
opt_output = Out.get(4)